# Probe evaluation — 0. Train the probe on the open base model

| Actor | Email | Role |
|-------|-------|------|
| **Probe owner** | `BENCHMARK_OWNER_EMAIL` | Owns the private linear probe and the prompts. Runs this notebook, then notebook 1. The enclave's release pins the benchmark-owner account as a data owner, so that is the account the probe owner uses |
| **Model owner** | `MODEL_OWNER_EMAIL` | Owns the private LoRA adapter. Takes no part in this notebook |
| **Enclave** | `ENCLAVE_EMAIL` | Tinfoil Container that runs the evaluation. Takes no part in this notebook |

The emails are set in notebooks 1 and 2. This notebook logs in nowhere and talks to no one.

**The setting.** A probe owner has a linear probe: a direction in a model's hidden states that
tracks a concept. They want to know what it reads on a fine-tune they cannot see. A model owner
built that fine-tune as a private LoRA adapter on an open base model, and is willing to be measured
but will not hand over the weights. The probe owner will not hand over the probe either.

What we, the probe owner, can see is the base model. So we train the probe here, on our own machine
or in Colab, on the open base model, and write it to `probe.json`. Notebook 1 then applies it inside
the enclave to base + adapter, which we never see.

The code that reads the hidden state and applies the probe is defined once below, as
`PROBE_READOUT_SRC`. It builds the training features here, runs a dry run on the base model in
notebook 1, and goes into the enclave job verbatim. That the same code runs in all three places is
the point: the only thing the enclave adds is the adapter.

**The concept is a placeholder.** Until a real probe owner brings their own probe and prompts, we
train one for a *specialised-advice request*: a prompt that asks for election, financial, health or
legal advice. We label prompts from the MLCommons AILuminate demo set by their hazard code. The four
`spc_*` hazards are the positives, and every other hazard is a negative.

A hazard code says what a prompt is about, so a probe that separates them may be picking up topic
words rather than anything deeper. Read the numbers at the end as a check that the mechanics work,
not as a finding about the model.

## Setup

In [ ]:
# transformers and accelerate at the versions the enclave job pins, so the
# features here go through the same code as there.
!uv pip install -q torch "transformers==4.57.6" "accelerate==1.12.0" scikit-learn

In [ ]:
import csv
import json
import random
import time
import types
from pathlib import Path

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# The open base model the model owner's adapter targets, at the commit the
# enclave job downloads (BASE_MODEL_REVISION in notebook 1).
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
BASE_MODEL_REVISION = "fe8a4ea1ffedaf415f4da2f062534de366a451e6"

# The placeholder concept, and the hazard codes that are its positives.
CONCEPT = "specialised-advice request"
POSITIVE_HAZARDS = {"spc_ele", "spc_fin", "spc_hlt", "spc_lgl"}

# All the positives, and a seeded sample of this many negatives per positive.
NEGATIVES_PER_POSITIVE = 3
# None trains on that whole sample. A number, such as 40, takes a stratified
# subsample of it instead, for a quick run.
MAX_PROMPTS = None
HELD_OUT_FRACTION = 0.25
THRESHOLD = 0.5
SEED = 0

AILUMINATE_CSV_URL = (
    "https://raw.githubusercontent.com/mlcommons/ailuminate/main/"
    "airr_official_1.0_demo_en_us_prompt_set_release.csv"
)
BENCH_URL = "https://raw.githubusercontent.com/OpenMined/double-blind-eval-bench/main/bench"

## Step 1 — Get the prompts

The [MLCommons AILuminate](https://github.com/mlcommons/ailuminate) demo prompt set has 1,200
prompts across 15 hazard codes: 100 for each of eleven hazards, and 100 shared among the four
`spc_*` hazards. Its columns are `release_prompt_id`, `prompt_text`, `hazard`, `persona`, `locale`
and `prompt_hash`.

Notebook 1 evaluates on ten prompts from the same set: the five private and five mock prompts in
[OpenMined/double-blind-eval-bench](https://github.com/OpenMined/double-blind-eval-bench). We leave
every one of them out of training, matched both by id (the bench's `prompt_uid` is AILuminate's
`release_prompt_id`) and by text, so the probe is never read on a prompt it was fitted to.

In [ ]:
!mkdir -p train_prompts
!curl -sfL "{AILUMINATE_CSV_URL}" -o train_prompts/ailuminate_demo.csv
!curl -sfL "{BENCH_URL}/dbe_prompts.csv" -o train_prompts/dbe_prompts.csv
!curl -sfL "{BENCH_URL}/dbe_prompts_mock.csv" -o train_prompts/dbe_prompts_mock.csv

In [ ]:
def read_csv(path: str) -> list[dict]:
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


demo = read_csv("train_prompts/ailuminate_demo.csv")
missing = {"release_prompt_id", "prompt_text", "hazard"} - set(demo[0])
assert not missing, f"The AILuminate file has no {sorted(missing)} column"
bench = read_csv("train_prompts/dbe_prompts.csv") + read_csv("train_prompts/dbe_prompts_mock.csv")

# Leave out every prompt notebook 1 evaluates on, matched by id and by text.
bench_ids = {r["prompt_uid"] for r in bench}
bench_texts = {r["prompt_text"].strip() for r in bench}
excluded = {
    r["release_prompt_id"]
    for r in demo
    if r["release_prompt_id"] in bench_ids or r["prompt_text"].strip() in bench_texts
}
pool = [r for r in demo if r["release_prompt_id"] not in excluded]

print(f"  AILuminate demo set : {len(demo)} prompts, {len({r['hazard'] for r in demo})} hazard codes")
print(f"  Evaluation prompts  : {len(bench)}")
print(f"  Excluded            : {len(excluded)}, leaving {len(pool)}")
if len(excluded) != len(bench):
    print(" 🟠 Not every evaluation prompt was found in the demo set. Check both files before training.")

## Step 2 — Label and sample

Every `spc_*` prompt is a positive. The negatives are a seeded random sample of the other hazards,
`NEGATIVES_PER_POSITIVE` for each positive, or all of them if there are not that many.
`class_weight="balanced"` in training makes up for the imbalance. On the demo set that is 97
positives and 291 negatives, 388 prompts in all.

In [ ]:
rng = random.Random(SEED)
positives = [r for r in pool if r["hazard"] in POSITIVE_HAZARDS]
negatives = [r for r in pool if r["hazard"] not in POSITIVE_HAZARDS]
n_negatives = min(len(negatives), NEGATIVES_PER_POSITIVE * len(positives))
if n_negatives < NEGATIVES_PER_POSITIVE * len(positives):
    print(
        f" 🟠 {len(negatives)} negatives is fewer than {NEGATIVES_PER_POSITIVE} per positive,"
        " so we use all of them."
    )

sample = [(r, 1) for r in positives] + [(r, 0) for r in rng.sample(negatives, n_negatives)]
if MAX_PROMPTS is not None and MAX_PROMPTS < len(sample):
    sample, _ = train_test_split(
        sample, train_size=MAX_PROMPTS, stratify=[y for _, y in sample], random_state=SEED
    )
rng.shuffle(sample)
rows = [r for r, _ in sample]
labels = np.array([y for _, y in sample])

print(f"  Positives : {labels.sum():>4}  ({', '.join(sorted(POSITIVE_HAZARDS))})")
print(f"  Negatives : {len(labels) - labels.sum():>4}  (the other hazard codes)")

## Step 3 — Load the open base model

At the pinned commit, in bfloat16, on CPU: the way the enclave job loads it. A probe is only as good
as the match between the hidden states it is trained on and the ones it is applied to, so we change
nothing here that the enclave does not.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_MODEL_REVISION,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
HIDDEN_SIZE = model.config.hidden_size
print(f"  {BASE_MODEL}: {N_LAYERS} layers, hidden size {HIDDEN_SIZE}")

## Step 4 — The readout

`PROBE_READOUT_SRC` is the one definition of how a probe reads the model. It is written as source so
that it can travel: Step 8 saves it as `probe_readout.py`, and notebook 1 reads that file back, runs
it on the base model for its dry run, and puts it verbatim into the job that the model owner reviews
and the enclave runs.

It takes the hidden state of the prompt's last token, after the chat template has opened the
assistant's turn: what the model has read just before it starts to answer. `hidden_states[0]` is the
embeddings and `hidden_states[i]` is the output of layer `i`. The probe is a dot product and a
sigmoid on that vector, in float32.

`readout.last_token_hiddens` reads several layers in one forward pass, which is how we try several layers
here for the price of one. `last_token_hidden` is the one-layer case that the job uses.

In [ ]:
PROBE_READOUT_SRC = '''\
"""The probe readout: a linear probe on one hidden state of a causal LM.

The same source builds the probe's training features on the open base model
(notebook 0), runs the probe owner's dry run on that base model (notebook 1),
and runs in the enclave on base + adapter (the job). It writes nothing: the
hidden states stay in memory, and only verdicts are returned.

Index convention: hidden_states[0] is the embeddings and hidden_states[i] is
the output of layer i, so a probe's layer runs from 0 to num_hidden_layers.
"""

import math

import torch

PROBE_FORMAT = "linear-probe/1"
PROBE_POSITION = "last_token_after_chat_template"
PROBE_KEYS = (
    "format", "concept", "base_model", "base_model_revision", "layer",
    "position", "hidden_size", "weight", "bias", "threshold", "mock",
)


def last_token_hiddens(model, tokenizer, prompt, layers):
    """The prompt's last-token hidden state at each of *layers*, as float32 rows.

    One forward pass over the prompt in the chat template, with the assistant's
    turn opened: what the model has read just before it starts to answer.
    """
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    return torch.stack([out.hidden_states[layer][0, -1] for layer in layers]).float()


def last_token_hidden(model, tokenizer, prompt, layer):
    """The prompt's last-token hidden state at *layer*, as a float32 vector."""
    return last_token_hiddens(model, tokenizer, prompt, [layer])[0]


def check_probe(probe, model):
    """Fail unless *probe* is a complete linear-probe/1 probe that fits *model*."""
    missing = [key for key in PROBE_KEYS if key not in probe]
    if missing:
        raise ValueError(f"The probe has no {missing}")
    if probe["format"] != PROBE_FORMAT or probe["position"] != PROBE_POSITION:
        raise ValueError(f"Not a {PROBE_FORMAT} probe read at {PROBE_POSITION}")
    hidden_size = model.config.hidden_size
    if probe["hidden_size"] != hidden_size or len(probe["weight"]) != hidden_size:
        raise ValueError(f"The probe does not fit the model's hidden size, {hidden_size}")
    layer = probe["layer"]
    if type(layer) is not int or not 0 <= layer <= model.config.num_hidden_layers:
        raise ValueError(f"Layer {layer!r} is not an index into hidden_states")
    numbers = [*probe["weight"], probe["bias"], probe["threshold"]]
    if not all(type(x) in (int, float) and math.isfinite(x) for x in numbers):
        raise ValueError("The probe's weight, bias and threshold must be finite numbers")


def probe_score(hidden, probe):
    """sigmoid(hidden @ weight + bias), in float32."""
    weight = torch.tensor(probe["weight"], dtype=torch.float32)
    return torch.sigmoid(hidden.float() @ weight + probe["bias"]).item()


def probe_readout(model, tokenizer, probe, rows, release_scores):
    """One verdict per row: prompt_uid, hazard, flagged, and score if release_scores."""
    check_probe(probe, model)
    verdicts = []
    for row in rows:
        hidden = last_token_hidden(model, tokenizer, row["prompt_text"], probe["layer"])
        score = probe_score(hidden, probe)
        verdict = {
            "prompt_uid": row["prompt_uid"],
            "hazard": row["hazard"],
            "flagged": score >= probe["threshold"],
        }
        if release_scores:
            verdict["score"] = score
        verdicts.append(verdict)
    return verdicts
'''
# Run it as a module of its own, the way notebook 1 runs probe_readout.py.
readout = types.ModuleType("probe_readout")
exec(PROBE_READOUT_SRC, readout.__dict__)

# A few layers spread across the middle of the stack, as indices into hidden_states.
CANDIDATE_LAYERS = sorted({round(N_LAYERS * f) for f in (0.25, 0.375, 0.5, 0.625, 0.75)})
print(f"  Candidate layers: {CANDIDATE_LAYERS} of {N_LAYERS}")

## Step 5 — Read the hidden states

One forward pass per prompt. We time the first few and print an estimate for the rest before
starting. On an Apple-silicon laptop CPU the 388 prompts took about 12 minutes, 1.9 seconds each;
Colab's two CPUs are slower. Setting `MAX_PROMPTS` above trades it for a smaller sample.

In [ ]:
def features(row: dict) -> np.ndarray:
    """The prompt's last-token hidden state at every candidate layer."""
    return readout.last_token_hiddens(model, tokenizer, row["prompt_text"], CANDIDATE_LAYERS).numpy()


TIMED = min(5, len(rows))
start = time.time()
X = [features(row) for row in rows[:TIMED]]
per_prompt = (time.time() - start) / TIMED
print(f"  {per_prompt:.2f} s per prompt: about {per_prompt * len(rows) / 60:.1f} minutes for all {len(rows)}")

In [ ]:
X = list(X)
start = time.time()
for i, row in enumerate(rows[len(X):], start=len(X) + 1):
    X.append(features(row))
    if i % 100 == 0 or i == len(rows):
        print(f"  {i}/{len(rows)}  ({time.time() - start:.0f} s)")
# (prompts, candidate layers, hidden size), in float32.
X = np.stack(X)

## Step 6 — Pick a layer and fit the probe

We hold out a quarter of the prompts, stratified by label, and leave them alone until Step 7. On the
rest, we score each candidate layer by its 5-fold cross-validated AUROC and keep the best one.
Choosing the layer on the held-out prompts and then reporting on them would flatter the result.

The probe is a logistic regression on standardised features, with `class_weight="balanced"`. We fold
the standardisation into the weight and the bias, so the enclave only computes a dot product, and
check that the folded probe scores exactly as the fitted one does.

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(rows)), test_size=HELD_OUT_FRACTION, stratify=labels, random_state=SEED
)
y_train, y_test = labels[idx_train], labels[idx_test]
print(f"  Train    : {len(y_train):>4} prompts, {y_train.sum():>3} positive")
print(f"  Held out : {len(y_test):>4} prompts, {y_test.sum():>3} positive")
print()


def make_classifier():
    return make_pipeline(
        StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=5000)
    )


def choose_layer(y: np.ndarray) -> tuple[int, dict]:
    """The candidate layer with the best cross-validated AUROC on the training split."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    aurocs = {
        layer: cross_val_score(make_classifier(), X[idx_train, i], y, cv=cv, scoring="roc_auc").mean()
        for i, layer in enumerate(CANDIDATE_LAYERS)
    }
    return max(aurocs, key=aurocs.get), aurocs


def probe_file(layer: int, weight: np.ndarray, bias: float, mock: bool) -> dict:
    """A probe in the linear-probe/1 format that probe_readout reads."""
    return {
        "format": readout.PROBE_FORMAT,
        "concept": CONCEPT,
        "base_model": BASE_MODEL,
        "base_model_revision": BASE_MODEL_REVISION,
        "layer": layer,
        "layer_convention": "hidden_states[0] is the embeddings; hidden_states[i] is the output of layer i",
        "position": readout.PROBE_POSITION,
        "hidden_size": HIDDEN_SIZE,
        "weight": np.asarray(weight, dtype=np.float32).tolist(),
        "bias": float(bias),
        "threshold": THRESHOLD,
        "mock": mock,
    }


def held_out_scores(probe: dict) -> np.ndarray:
    """The probe's score on every held-out prompt, through the shared probe_score."""
    i = CANDIDATE_LAYERS.index(probe["layer"])
    return np.array([readout.probe_score(torch.from_numpy(X[j, i]), probe) for j in idx_test])


def fit_probe(layer: int, y: np.ndarray) -> dict:
    """Fit on the training split at *layer*, with the standardisation folded in."""
    i = CANDIDATE_LAYERS.index(layer)
    fitted = make_classifier().fit(X[idx_train, i], y)
    scaler, classifier = fitted[0], fitted[-1]
    # w·(x - mean)/scale + b  =  (w/scale)·x + (b - (w/scale)·mean)
    weight = classifier.coef_[0] / scaler.scale_
    bias = classifier.intercept_[0] - weight @ scaler.mean_
    probe = probe_file(layer, weight, bias, mock=False)
    expected = fitted.predict_proba(X[idx_test, i])[:, 1]
    np.testing.assert_allclose(held_out_scores(probe), expected, atol=1e-4)
    return probe


layer, aurocs = choose_layer(y_train)
for candidate, auroc in aurocs.items():
    chosen = "  ← chosen" if candidate == layer else ""
    print(f"  layer {candidate:>2}: cross-validated AUROC {auroc:.3f}{chosen}")
probe = fit_probe(layer, y_train)

## Step 7 — How well it does, and a control

On the held-out prompts, with the probe exactly as it will be written to `probe.json`: its AUROC, and
its balanced accuracy at the 0.5 threshold. The control runs the same pipeline (choose a layer, fit,
fold) on training labels shuffled at random, and scores it against the true held-out labels. It
should land near 0.5; if it does not, something leaks.

The held-out set is small: a quarter of the sample, so only about two dozen positives. One
misranked positive moves the AUROC by a few hundredths, so the table gives the counts next to it.

These are a few hundred prompts from one prompt set, labelled by topic. A high AUROC says that the
base model's hidden states separate these topics linearly, which is not surprising. It says nothing
yet about the fine-tune, which is what the enclave run is for.

In [ ]:
def evaluate(probe: dict) -> tuple[float, float]:
    scores = held_out_scores(probe)
    return roc_auc_score(y_test, scores), balanced_accuracy_score(y_test, scores >= probe["threshold"])


shuffled = np.random.default_rng(SEED).permutation(y_train)
control_layer, _ = choose_layer(shuffled)
control = fit_probe(control_layer, shuffled)

held_out = f"{y_test.sum()} of {len(y_test)}"
print()
print(f"  {'':<34}{'AUROC':>7}{'balanced accuracy':>20}{'held-out positives':>21}")
for name, p in [
    (f"probe, layer {layer}", probe),
    (f"shuffled-label control, layer {control_layer}", control),
]:
    auroc, balanced_accuracy = evaluate(p)
    print(f"  {name:<34}{auroc:>7.3f}{balanced_accuracy:>20.3f}{held_out:>21}")

In [ ]:
# Where the probe fires on the held-out prompts, by hazard code.
flagged = held_out_scores(probe) >= probe["threshold"]
hazards = np.array([rows[j]["hazard"] for j in idx_test])
for hazard in sorted(set(hazards), key=lambda h: (h not in POSITIVE_HAZARDS, h)):
    mask = hazards == hazard
    kind = "positive" if hazard in POSITIVE_HAZARDS else "negative"
    print(f"  {hazard:<8} {kind:<9} flagged {flagged[mask].sum():>3} of {mask.sum():>3}")

## Step 8 — Write the probe

Three files, next to this notebook:

- `probe.json`: the probe, in `linear-probe/1`. It holds the layer, the folded weight and bias, the
  threshold, and the base model and commit it was trained on. This is the private half: it goes to
  the enclave and nowhere else, so keep it out of version control. The folder's `.gitignore`
  already does.
- `probe_mock.json`: the same fields with seeded random weights, `"mock": true`, and a note saying
  it is a placeholder. This is the half the model owner can see: the concept and the layer, not the
  weights.
- `probe_readout.py`: `PROBE_READOUT_SRC`, as it is. Notebook 1 reads it back, so the code that reads
  the probe in the enclave is the code that trained it here.

In [ ]:
mock_weight = np.random.default_rng(SEED).normal(0.0, HIDDEN_SIZE**-0.5, HIDDEN_SIZE)
mock = probe_file(layer, mock_weight, 0.0, mock=True)
mock["note"] = (
    "Placeholder with seeded random weights, not the probe. "
    "The real probe is shared with the enclave only."
)
for p in (probe, mock):
    readout.check_probe(p, model)

Path("probe.json").write_text(json.dumps(probe, indent=2) + "\n")
Path("probe_mock.json").write_text(json.dumps(mock, indent=2) + "\n")
Path("probe_readout.py").write_text(PROBE_READOUT_SRC)
print("  Wrote probe.json, probe_mock.json and probe_readout.py")
print(f"  Probe: {CONCEPT!r}, layer {layer} of {BASE_MODEL}")

In [ ]:
# In Colab, download the three files here, then upload them in notebook 1.
# from google.colab import files
# for name in ("probe.json", "probe_mock.json", "probe_readout.py"):
#     files.download(name)

---

We have a probe trained on the open base model, and the code that reads it. Neither has met the
model owner's adapter yet. Notebook 1 runs the same code on the base model once more, as a dry run,
then sends the probe into the enclave, where the adapter is the only thing that changes.